In [ ]:
!pip install -q kagglehub
import kagglehub, shutil, os, glob

# Baixa
path = kagglehub.dataset_download("laotse/credit-risk-dataset")
print("Baixou em:", path)

# Acha o csv e copia pro /content com nome fixo
arquivos = glob.glob(os.path.join(path, "*.csv"))
print("Achou:", arquivos)

if arquivos:
    shutil.copy(arquivos[0], "/content/credit risk dataset.csv")
    print("✅ Copiado para /content/credit risk dataset.csv")

Using Colab cache for faster access to the 'credit-risk-dataset' dataset.
Baixou em: /kaggle/input/credit-risk-dataset
Achou: ['/kaggle/input/credit-risk-dataset/credit_risk_dataset.csv']
✅ Copiado para /content/credit risk dataset.csv


In [ ]:
from google.colab import files
files.download('/content/credit_risk_dataset.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
df = pd.read_csv('credit_risk_dataset.csv')
print(df.shape, df.isnull().sum())

(32581, 12) person_age                       0
person_income                    0
person_home_ownership            0
person_emp_length              895
loan_intent                      0
loan_grade                       0
loan_amnt                        0
loan_int_rate                 3116
loan_status                      0
loan_percent_income              0
cb_person_default_on_file        0
cb_person_cred_hist_length       0
dtype: int64


In [ ]:
df = df.drop_duplicates()
for col in df.select_dtypes(include='number').columns:
    df[col]=df[col].fillna(df[col].median())
df = df[(df['person_age']>=18)&(df['person_age']<=80)]

In [ ]:
df['comprometimento_renda'] = (df['loan_amnt']/df['person_income'])*100

In [ ]:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import joblib

# 1. CARREGA
df = pd.read_csv("/content/credit risk dataset.csv")
print(f"Base: {df.shape}")
print("Nulos:\n", df.isnull().sum())

# Fase 2: TRATAMENTO E LIMPEZA
# Duplicatas
print(f"Duplicatas: {df.duplicated().sum()}")
df = df.drop_duplicates()

# Valores Nulos - Justificativa: Mediana é mais robusta que Média para dados assimétricos de renda
# Tratamos nulos ANTES do calculo para evitar NaN ou infinito
for col in df.select_dtypes(include=[np.number]).columns:
    df[col] = df[col].fillna(df[col].median())
for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].fillna(df[col].mode()[0])

# Tratamento de Outliers - Decisão fundamentada
# KNN é extremamente sensivel a distancias euclidianas, Arvore é robusta
# Vamos fazer clipping para manter o dado mas reduzir impacto
df = df[(df['person_age'] >= 18) & (df['person_age'] <= 80)]
# Garbage In Garbage Out
print(f"Após limpeza: {df.shape}")

# Fase 3: FEATURE ENGINEERING - OBRIGATÓRIA DO EDITAL
df['comprometimento_renda'] = (df['loan_amnt'] / df['person_income']) * 100
# Extra para enriquecer
df['renda_por_idade'] = df['person_income'] / df['person_age']
print("Coluna obrigatoria criada: comprometimento_renda")

# Fase 4: SEPARAÇÃO, BALANCEAMENTO E ESCALONAMENTO SEGURO
X = df.drop('loan_status', axis=1)
y = df['loan_status']

cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

num_pipe = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
cat_pipe = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))])

preprocess = ColumnTransformer([('num', num_pipe, num_cols), ('cat', cat_pipe, cat_cols)])

# Fase 5: MODELAGEM COM BALANCEAMENTO (Regra de Ouro)
# SMOTE para corrigir desbalanceamento
knn_pipe = ImbPipeline([('preprocessor', preprocess), ('smote', SMOTE(random_state=42)), ('classifier', KNeighborsClassifier(n_neighbors=5))])
tree_pipe = ImbPipeline([('preprocessor', preprocess), ('smote', SMOTE(random_state=42)), ('classifier', DecisionTreeClassifier(max_depth=6, min_samples_leaf=10, random_state=42))])
rf_pipe = ImbPipeline([('preprocessor', preprocess), ('smote', SMOTE(random_state=42)), ('classifier', RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42))])

modelos = {'KNN': knn_pipe, 'Arvore Decisao': tree_pipe, 'RandomForest': rf_pipe}

for nome, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    acc_train = accuracy_score(y_train, modelo.predict(X_train))
    acc_test = accuracy_score(y_test, modelo.predict(X_test))
    print(f"{nome} -> Train: {acc_train:.4f} | Test: {acc_test:.4f} | Gap Overfitting: {acc_train-acc_test:.4f}")

# Fase 6: AVALIAÇÃO
melhor = rf_pipe
y_pred = melhor.predict(X_test)
print("\nAcurácia Final:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

joblib.dump(melhor, 'pipeline_credito.pkl')
print("\n✅ Pipeline salvo!")

Base: (32581, 12)
Nulos:
 person_age                       0
person_income                    0
person_home_ownership            0
person_emp_length              895
loan_intent                      0
loan_grade                       0
loan_amnt                        0
loan_int_rate                 3116
loan_status                      0
loan_percent_income              0
cb_person_default_on_file        0
cb_person_cred_hist_length       0
dtype: int64
Duplicatas: 165
Após limpeza: (32409, 12)
Coluna obrigatoria criada: comprometimento_renda
KNN -> Train: 0.8962 | Test: 0.8116 | Gap Overfitting: 0.0845
Arvore Decisao -> Train: 0.8897 | Test: 0.8897 | Gap Overfitting: -0.0000
RandomForest -> Train: 0.9322 | Test: 0.9175 | Gap Overfitting: 0.0147

Acurácia Final: 0.9174637457574822
              precision    recall  f1-score   support

           0       0.93      0.97      0.95      5064
           1       0.86      0.74      0.80      1418

    accuracy                           0.92